# Contact-Predicted SAAP Analysis

Tests whether contact-predicted SAAP peptides (proximal to destabilizing missense mutations) are:
1. **Detected more often** in TMT channels whose patient carries the corresponding destabilizing missense
2. **More abundant** (higher RI intensity) in carrier channels vs non-carrier channels

Compares destabilizing (AM+SPURS, SPURS only, AM only) vs neutral contact predictions as a negative control.

**Logic:**
- Contact FASTA entries tagged `destab_contact` or `neutral_contact` in the description
- For each detected contact PSM in a plex:
  - Parse gene + contact position from the entry name
  - Find which plex patients have a destabilizing missense within 3Å of that position
  - Compare RI intensity: carrier channels vs non-carrier channels
  - Compute detection rate per channel group

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# ── CONFIG ────────────────────────────────────────────────────────────────────
REPO_DIR     = '/home/leduc.an/AAS_Evo_project/AAS_Evo'
TMT_MAP      = f'{REPO_DIR}/metadata/PDC_meta/pdc_file_tmt_map.tsv'
GDC_META     = f'{REPO_DIR}/metadata/GDC_meta/gdc_meta_matched.tsv'
MISSENSE     = '/scratch/leduc.an/AAS_Evo/ANALYSIS/all_missense_with_spurs.tsv'
REF_FASTA    = '/scratch/leduc.an/AAS_Evo/SEQ_FILES/uniprot_human_canonical.fasta'
RESULTS_BASE = '/scratch/leduc.an/AAS_Evo/MS_SEARCH/results_contact'
CONTACT_DIR  = '/scratch/leduc.an/AAS_Evo/SPURS/contact_maps'
DDG_DIR      = '/scratch/leduc.an/AAS_Evo/SPURS/ddg_matrices'
PLEX_LIST    = '/scratch/leduc.an/AAS_Evo/MS_SEARCH/plex_list.txt'
OUT_DIR      = '/scratch/leduc.an/AAS_Evo/ANALYSIS/contact_saap'
os.makedirs(OUT_DIR, exist_ok=True)

AM_THRESHOLD    = 0.564
SPURS_THRESHOLD = 0.5
AM_BENIGN_MAX   = 0.1
GNOMAD_NEUTRAL  = 0.1
GNOMAD_MAX      = 0.01
VAF_THRESHOLD   = 0.3
DIST_THRESHOLD  = 3.0   # Å Cα-Cα (must match generate_contact_saap_fastas.py)

TMT_CHANNEL_MAP = {
    'tmt_126':'126','tmt_127n':'127N','tmt_127c':'127C',
    'tmt_128n':'128N','tmt_128c':'128C','tmt_129n':'129N',
    'tmt_129c':'129C','tmt_130n':'130N','tmt_130c':'130C',
    'tmt_131':'131N','tmt_131c':'131C','tmt_126c':'126C','tmt_134n':'134N',
}
CHANNEL_ORDER = ['126','127N','127C','128N','128C','129N','129C','130N','130C','131N','131C']


In [ ]:
# ── LOAD METADATA ─────────────────────────────────────────────────────────────
tmt = pd.read_csv(TMT_MAP, sep='\t')
gdc = pd.read_csv(GDC_META, sep='\t')
if 'file_id' in gdc.columns and 'gdc_file_id' not in gdc.columns:
    gdc = gdc.rename(columns={'file_id':'gdc_file_id'})

with open(PLEX_LIST) as f:
    plex_ids = [l.strip() for l in f if l.strip()]
# Only plexes with results_contact results
plex_ids = [p for p in plex_ids
            if os.path.exists(os.path.join(RESULTS_BASE, p, 'combined_protein.tsv'))]

uuid_to_stype = gdc.set_index('gdc_file_id')['sample_type'].to_dict()
case_sample_to_uuid = gdc.set_index(['case_submitter_id','sample_type'])['gdc_file_id'].to_dict()

print(f'Plexes with results_contact: {len(plex_ids)}')

In [ ]:
# ── LOAD & FILTER MISSENSE TABLE ──────────────────────────────────────────────
print('Loading missense table...')
miss = pd.read_csv(MISSENSE, sep='\t', low_memory=False,
                   usecols=['sample_id','SYMBOL','Protein_position',
                            'Amino_acids','VAF','gnomADe_AF',
                            'am_pathogenicity','spurs_ddg'])
miss['VAF']             = pd.to_numeric(miss['VAF'],             errors='coerce')
miss['gnomADe_AF']      = pd.to_numeric(miss['gnomADe_AF'],      errors='coerce').fillna(0)
miss['am_pathogenicity']= pd.to_numeric(miss['am_pathogenicity'],errors='coerce')
miss['spurs_ddg']       = pd.to_numeric(miss['spurs_ddg'],       errors='coerce')
miss['pos']             = pd.to_numeric(
    miss['Protein_position'].astype(str).str.split('-').str[0], errors='coerce')

base_ok   = (miss['VAF'] >= VAF_THRESHOLD) & (miss['gnomADe_AF'] < GNOMAD_MAX)
am_pos    = miss['am_pathogenicity'] >= AM_THRESHOLD
spurs_pos = miss['spurs_ddg']        >= SPURS_THRESHOLD

def classify_miss(am, spurs):
    a = pd.notna(am)    and am    >= AM_THRESHOLD
    s = pd.notna(spurs) and spurs >= SPURS_THRESHOLD
    if a and s:  return 'Both AM+SPURS'
    if s:        return 'SPURS only'
    if a:        return 'AM only'
    if pd.notna(am) and am <= AM_BENIGN_MAX: return 'Neutral'
    return None

destab  = miss[(am_pos | spurs_pos) & base_ok].copy()
neutral = miss[(miss['am_pathogenicity'] <= AM_BENIGN_MAX) &
               (miss['gnomADe_AF'] >= GNOMAD_NEUTRAL)].copy()

destab['group']  = destab.apply(lambda r: classify_miss(r['am_pathogenicity'], r['spurs_ddg']), axis=1)
neutral['group'] = 'Neutral'
all_miss = pd.concat([destab, neutral], ignore_index=True)
all_miss_indexed = all_miss.groupby('sample_id')

all_processed_uuids = set(miss['sample_id'].unique())
print(f'Destab: {len(destab):,} | Neutral: {len(neutral):,}')

In [ ]:
# ── CONTACT MAP HELPERS + SEQUENCE LOADING ────────────────────────────────────
import re as _re

gene_to_acc = {f.name.split('.')[1]: f.name.split('.')[0]
               for f in Path(DDG_DIR).glob('*.ddg_matrix.tsv')}

print('Loading reference sequences...')
seqs = {}
cur_acc = None
with open(REF_FASTA) as fh:
    for line in fh:
        line = line.rstrip()
        if line.startswith('>'):
            parts = line.split('|')
            cur_acc  = parts[1] if len(parts) >= 3 else line[1:].split()[0]
            m_gene   = _re.search(r'GN=(\S+)', line)
            cur_gene = m_gene.group(1) if m_gene else None
            seqs[cur_acc] = []
            if cur_gene and cur_gene not in gene_to_acc:
                gene_to_acc[cur_gene] = cur_acc
        elif cur_acc:
            seqs[cur_acc].append(line)
seqs = {acc: ''.join(s) for acc, s in seqs.items()}
print(f'  {len(seqs):,} sequences | {len(gene_to_acc):,} gene->acc mappings')

# ── Tryptic digestion ─────────────────────────────────────────────────────────
def _tryptic_cuts(seq):
    cuts = [-1]
    for i, aa in enumerate(seq):
        if aa in 'KR' and (i + 1 >= len(seq) or seq[i + 1] != 'P'):
            cuts.append(i)
    cuts.append(len(seq) - 1)
    return cuts

def get_wt_peptides(acc, contact_pos_1based):
    """Return WT tryptic peptide sequences (0- and 1-missed cleavage) covering contact_pos."""
    seq = seqs.get(acc, '')
    if not seq or contact_pos_1based < 1 or contact_pos_1based > len(seq):
        return []
    cuts = _tryptic_cuts(seq)
    idx = contact_pos_1based - 1
    results = set()
    for i in range(len(cuts) - 1):
        for j in range(i + 1, min(i + 3, len(cuts))):
            start = cuts[i] + 1
            end   = cuts[j] + 1
            if start <= idx < end:
                pep = seq[start:end]
                if len(pep) >= 6:
                    results.add(pep)
    return list(results)

# ── Contact map helpers ───────────────────────────────────────────────────────
cm_cache     = {}
nearby_cache = {}

def load_contact_map(acc):
    cdir = Path(CONTACT_DIR)
    for npy_path in sorted(cdir.glob(f'AF-{acc}-*F1.npy')):
        csv_path = npy_path.with_suffix('.csv')
        if not csv_path.exists(): continue
        try:
            meta = pd.read_csv(csv_path, index_col=0)
            dm   = np.load(npy_path)
            p2i  = {int(row['id']): i for i, row in meta.iterrows() if pd.notna(row['id'])}
            return p2i, dm
        except Exception:
            continue
    return None, None

def get_nearby(acc, pos):
    key = (acc, pos)
    if key not in nearby_cache:
        if acc not in cm_cache:
            cm_cache[acc] = load_contact_map(acc)
        p2i, dm = cm_cache[acc]
        if dm is None:
            nearby_cache[key] = []
        else:
            idx = p2i.get(pos)
            if idx is None:
                nearby_cache[key] = []
            else:
                pos_arr = np.array(list(p2i.keys()),   dtype=np.int32)
                idx_arr = np.array(list(p2i.values()), dtype=np.int32)
                d_vals  = dm[idx][idx_arr]
                mask    = (d_vals > 0) & (d_vals < DIST_THRESHOLD) & (pos_arr != pos)
                nearby_cache[key] = pos_arr[mask].tolist()
    return nearby_cache[key]

def find_ri_cols(df):
    found = {}
    for ch in CHANNEL_ORDER + ['126C','132N','132C','133N','134N']:
        if ch in df.columns:
            found[ch] = ch; continue
        cands = [c for c in df.columns if c.startswith('Intensity') and c.endswith(f'_{ch}')]
        if cands: found[ch] = cands[0]
    return found

def get_channel_uuid_map(plex_id):
    pt = tmt[tmt['run_metadata_id'] == plex_id][['tmt_channel','case_submitter_id','sample_type']].drop_duplicates()
    pt = pt[~pt['case_submitter_id'].str.lower().isin(['ref','reference','pooled','pool','nan'])]
    pt['channel'] = pt['tmt_channel'].map(TMT_CHANNEL_MAP)
    pt = pt.dropna(subset=['channel'])
    merged = pt.merge(gdc[['gdc_file_id','case_submitter_id','sample_type']],
                      on=['case_submitter_id','sample_type'], how='left')
    return merged.dropna(subset=['gdc_file_id']).drop_duplicates('channel').set_index('channel')['gdc_file_id'].to_dict()

print('Helpers defined')


In [ ]:
# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
# For each contact-predicted swap peptide detected in a plex:
#   1. Find the corresponding WT tryptic peptide
#   2. Compute log2(swap_RI / wt_RI) per channel  (self-normalizing — no scale needed)
#   3. Label each channel as carrier or non-carrier of nearby destab missense
records = []
n_done = n_no_psm = n_no_wt = 0

for pi, plex_id in enumerate(plex_ids):
    if pi % 5 == 0:
        print(f'  {pi}/{len(plex_ids)} plexes, {len(records):,} records', flush=True)

    psm_files = sorted(glob.glob(os.path.join(RESULTS_BASE, plex_id, '*_1/psm.tsv')))
    if not psm_files:
        psm_files = sorted(glob.glob(os.path.join(RESULTS_BASE, plex_id, 'psm.tsv')))
    if not psm_files:
        n_no_psm += 1; continue

    psm = pd.read_csv(psm_files[0], sep='\t', low_memory=False)
    ri_col_map = find_ri_cols(psm)
    if not ri_col_map: continue

    ch_uuid = get_channel_uuid_map(plex_id)
    channels_with_genomics = {ch for ch, uuid in ch_uuid.items()
                              if uuid in all_processed_uuids}
    if len(channels_with_genomics) < 2: continue

    # Aggregate RI per peptide sequence across all PSMs (sum across spectra)
    ri_cols_list = list(ri_col_map.values())
    pep_ri = (psm.groupby('Peptide')[ri_cols_list]
                 .sum()
                 .rename(columns={v: k for k, v in ri_col_map.items()}))

    # Get plex patient missense
    plex_uuids = set(ch_uuid.values())
    plex_miss_frames = [all_miss_indexed.get_group(uid)
                        for uid in plex_uuids if uid in all_miss_indexed.groups]
    plex_miss = pd.concat(plex_miss_frames) if plex_miss_frames else pd.DataFrame()
    if len(plex_miss) == 0: continue

    # Contact PSMs
    contact_mask = psm['Entry Name'].astype(str).str.endswith('-contact')
    contact_psm  = psm[contact_mask].copy()
    if len(contact_psm) == 0: continue

    # Deduplicate: one entry per unique (Entry Name, Protein ID, Peptide)
    seen = set()
    for _, row in contact_psm.iterrows():
        entry_name = str(row.get('Entry Name', ''))
        protein_id  = str(row.get('Protein ID', ''))
        pep_seq     = str(row.get('Peptide', ''))

        dedup_key = (entry_name, protein_id, pep_seq)
        if dedup_key in seen: continue
        seen.add(dedup_key)

        # Parse gene from Entry Name: {GENE}-contact
        m_gene = _re.match(r'^([A-Z0-9]+)-contact$', entry_name)
        if not m_gene: continue
        gene = m_gene.group(1)

        # Parse swap from Protein ID: {ACC}-{WT}{pos}{ALT}-{HASH}
        m_swap = _re.search(r'-([A-Z])(\d+)([A-Z])-[A-Z0-9]{4}$', protein_id)
        if not m_swap: continue
        wt_aa, contact_pos, alt_aa = m_swap.group(1), int(m_swap.group(2)), m_swap.group(3)

        acc = gene_to_acc.get(gene)
        if not acc: continue

        # Find WT peptide(s) covering this position
        wt_peps = get_wt_peptides(acc, contact_pos)
        wt_pep  = next((p for p in wt_peps if p in pep_ri.index), None)
        if wt_pep is None:
            n_no_wt += 1; continue

        # Skip if swap not in pep_ri (shouldn't happen, but guard)
        if pep_seq not in pep_ri.index: continue

        # Find carrier UUIDs via nearby missense in plex patients
        gene_miss = plex_miss[plex_miss['SYMBOL'] == gene].dropna(subset=['pos'])
        if len(gene_miss) == 0: continue

        carrier_uuids = set()
        source_group  = None
        for miss_pos in gene_miss['pos'].unique():
            nearby = get_nearby(acc, int(miss_pos))
            if contact_pos in nearby:
                matched = gene_miss[gene_miss['pos'] == miss_pos]
                carrier_uuids |= set(matched['sample_id'])
                grp = matched['group'].mode()
                if len(grp): source_group = grp.iloc[0]

        if not carrier_uuids or source_group is None: continue

        carrier_chs    = [ch for ch in channels_with_genomics
                          if ch_uuid.get(ch) in carrier_uuids and ch in ri_col_map]
        noncarrier_chs = [ch for ch in channels_with_genomics
                          if ch_uuid.get(ch) not in carrier_uuids and ch in ri_col_map]
        if not carrier_chs or len(noncarrier_chs) < 2: continue

        # Compute log2(swap_RI / wt_RI) per channel — ratio is self-normalizing
        swap_row = pep_ri.loc[pep_seq]
        wt_row   = pep_ri.loc[wt_pep]

        for ch in carrier_chs + noncarrier_chs:
            swap_ri = swap_row.get(ch, 0)
            wt_ri   = wt_row.get(ch, 0)
            if wt_ri <= 0: continue  # WT not quantified in this channel
            raas = swap_ri / wt_ri
            log2_raas = float(np.log2(raas)) if raas > 0 else np.nan
            records.append({
                'plex_id':      plex_id,
                'gene':         gene,
                'contact_pos':  contact_pos,
                'swap':         f'{wt_aa}{contact_pos}{alt_aa}',
                'wt_peptide':   wt_pep,
                'channel':      ch,
                'source_group': source_group,
                'is_carrier':   ch in carrier_chs,
                'swap_ri':      float(swap_ri),
                'wt_ri':        float(wt_ri),
                'log2_raas':    log2_raas,
            })
    n_done += 1

df = pd.DataFrame(records)
print(f'Done: {n_done} plexes | {len(df):,} channel records | {n_no_wt} swap/no-WT skips')
if len(df):
    print(df['source_group'].value_counts())
    print(f"Carrier rows: {df['is_carrier'].sum():,} | Non-carrier: {(~df['is_carrier']).sum():,}")


In [ ]:
# ── GLOBAL STATS ──────────────────────────────────────────────────────────────
DESTAB_GROUPS  = ['Both AM+SPURS', 'SPURS only', 'AM only']
NEUTRAL_GROUPS = ['Neutral']

carrier_df    = df[df['is_carrier']  & df['log2_raas'].notna()]
noncarrier_df = df[~df['is_carrier'] & df['log2_raas'].notna()]

print(f'Total records with log2_raas: {df["log2_raas"].notna().sum():,}')
print(f'  Carrier channels:     {len(carrier_df):,}')
print(f'  Non-carrier channels: {len(noncarrier_df):,}')
print()
print('source_group breakdown (carrier channels):')
print(carrier_df['source_group'].value_counts().to_string())

print()
print('── log2(swap / WT) per group ───────────────────────────────────────────')
for grp in DESTAB_GROUPS + NEUTRAL_GROUPS:
    c = carrier_df[carrier_df['source_group'] == grp]['log2_raas']
    n = noncarrier_df[noncarrier_df['source_group'] == grp]['log2_raas']
    if len(c) < 3 or len(n) < 3: continue
    u, p_mw = stats.mannwhitneyu(c, n, alternative='greater')
    print(f'{grp:<20}  carrier n={len(c):,} mean={c.mean():.3f} | '
          f'non-carrier n={len(n):,} mean={n.mean():.3f} | MW p={p_mw:.2e}')

# Overall destab vs neutral in carriers
c_destab  = carrier_df[carrier_df['source_group'].isin(DESTAB_GROUPS)]['log2_raas']
c_neutral = carrier_df[carrier_df['source_group'].isin(NEUTRAL_GROUPS)]['log2_raas']
if len(c_destab) >= 3 and len(c_neutral) >= 3:
    u, p = stats.mannwhitneyu(c_destab, c_neutral, alternative='greater')
    print(f'\nDestab carriers vs Neutral carriers: MW p={p:.2e}  '
          f'(n={len(c_destab):,} vs {len(c_neutral):,})')


In [ ]:
# ── PLOTS ─────────────────────────────────────────────────────────────────────
import math
DESTAB_GROUPS  = ['Both AM+SPURS', 'SPURS only', 'AM only']
NEUTRAL_GROUPS = ['Neutral']

GROUP_SPECS = [
    ('Both AM+SPURS', '#d62728'),
    ('SPURS only',    '#ff7f0e'),
    ('AM only',       '#9467bd'),
    ('Neutral',       '#2ca02c'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Panel A: violin of log2_raas — carrier vs non-carrier per group ────────────
ax = axes[0]
positions, violin_data, violin_colors, xtick_labels = [], [], [], []
pos = 0
for grp, color in GROUP_SPECS:
    for label, is_c in [('carrier', True), ('non-carrier', False)]:
        sub = df[(df['source_group'] == grp) & (df['is_carrier'] == is_c)]['log2_raas'].dropna()
        if len(sub) < 3: pos += 1; continue
        positions.append(pos)
        violin_data.append(sub.values)
        violin_colors.append(color if is_c else '#aaaaaa')
        xtick_labels.append((pos, f'{grp}\n{label}\nn={len(sub):,}'))
        pos += 1
    pos += 0.5  # gap between groups

if violin_data:
    parts = ax.violinplot(violin_data, positions=positions,
                          showmedians=True, showextrema=False)
    for pc, col in zip(parts['bodies'], violin_colors):
        pc.set_facecolor(col); pc.set_alpha(0.65)
    parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(2)

ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xticks([x for x, _ in xtick_labels])
ax.set_xticklabels([l for _, l in xtick_labels], fontsize=6.5, rotation=30, ha='right')
ax.set_ylabel('log2(swap RI / WT RI)')
ax.set_title('RAAS-like ratio: swap / WT peptide\nper channel (carrier vs non-carrier)')

# ── Panel B: mean log2_raas — carrier vs non-carrier, destab combined vs neutral ──
ax2 = axes[1]
combos = [
    ('Destab carrier',      df[df['source_group'].isin(DESTAB_GROUPS) &  df['is_carrier']]['log2_raas'].dropna(), '#d62728'),
    ('Destab non-carrier',  df[df['source_group'].isin(DESTAB_GROUPS) & ~df['is_carrier']]['log2_raas'].dropna(), '#fa9898'),
    ('Neutral carrier',     df[df['source_group'].isin(NEUTRAL_GROUPS) &  df['is_carrier']]['log2_raas'].dropna(), '#2ca02c'),
    ('Neutral non-carrier', df[df['source_group'].isin(NEUTRAL_GROUPS) & ~df['is_carrier']]['log2_raas'].dropna(), '#98d698'),
]
for i, (label, d, color) in enumerate(combos):
    if len(d) == 0: continue
    t, p = stats.ttest_1samp(d, popmean=0) if len(d) >= 3 else (float('nan'), float('nan'))
    ax2.bar(i, d.mean(), yerr=d.sem(), capsize=5, color=color, alpha=0.9,
            error_kw={'linewidth':1.5})
    pstr = f'p={p:.3f}' if not math.isnan(p) else ''
    ax2.text(i, d.mean() + d.sem() + 0.005,
             f'n={len(d):,}\n{pstr}', ha='center', va='bottom', fontsize=7.5)
ax2.axhline(0, color='k', lw=0.8, ls='--')
ax2.set_xticks(range(len(combos)))
ax2.set_xticklabels([l for l, _, _ in combos], fontsize=8, rotation=20, ha='right')
ax2.set_ylabel('mean log2(swap RI / WT RI)')
ax2.set_title('Mean RAAS ratio by carrier status\n(t-test vs 0)')

plt.suptitle('Contact-predicted SAAP: RAAS ratio in missense carrier vs non-carrier channels',
             fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'contact_saap_raas.pdf'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── SAVE ──────────────────────────────────────────────────────────────────────
df.to_csv(os.path.join(OUT_DIR, 'contact_saap_records.tsv'), sep='\t', index=False)
print(f'Saved to {OUT_DIR}')